# 05 — Composite Elite Label Construction

This notebook creates the target variable used in the modelling stage. It combines three signals and applies a **2-of-3 rule**:

1. **Market-value growth:** top 20% of two-year value-growth ratios within primary position.
2. **Club-tier progression:** movement to a club in a higher average-squad-value quartile.
3. **Performance breakout:** position-specific composite score in the top 20% within position.

**Important:** the market-value signal is percentile-based in the implemented code; it is not a fixed 2.0x growth threshold. The thresholds are relative to this study sample and are not universal definitions of elite potential.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project root whether this notebook is launched from the
# repository root or from the notebooks/ directory.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DATA_DIR / "combined_with_valuations.csv"
VALUATIONS_FILE = RAW_DATA_DIR / "transfermarkt" / "player_valuations.csv"
OUTPUT_FILE = PROCESSED_DATA_DIR / "final_labeled_dataset.csv"

for required_file in [INPUT_FILE, VALUATIONS_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file not found: {required_file}. "
            "See data/README.md for the expected data locations."
        )

print(f"Project root: {PROJECT_ROOT}")

Project root: /mnt/data/football-player-development-prediction-repro


In [2]:
# The imports and project paths are configured in the setup cell above.

## 1. Load the final matched + valued dataset

In [3]:
final_df = pd.read_csv(INPUT_FILE)
print(f"Shape before deduplication: {final_df.shape}")

required_columns = [
    "Player", "Season", "Pos", "value_at_observation",
    "value_2yr_later", "club_at_observation", "club_2yr_later"
]
missing_columns = [column for column in required_columns if column not in final_df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

# Protect against accidental duplicate player-season records upstream.
duplicate_count = final_df.duplicated(subset=["Player", "Season"]).sum()
if duplicate_count:
    print(f"Warning: removing {duplicate_count} duplicate Player/Season rows.")
    final_df = final_df.drop_duplicates(subset=["Player", "Season"], keep="first")

print(f"Shape after deduplication: {final_df.shape}")

Shape before deduplication: (2872, 296)
Shape after deduplication: (2872, 296)


## 2. Signal 1 — Market value growth

In [4]:
final_df['value_growth_ratio'] = final_df['value_2yr_later'] / final_df['value_at_observation']

# Position-relative percentile threshold rather than a flat ratio for
# everyone. A fixed 2x cutoff would systematically favour forwards (higher,
# more volatile valuations) and players starting from a lower valuation
# base (e.g. smaller-league origin clubs), since the same absolute
# improvement produces a bigger ratio purely from starting lower. This
# also keeps Signal 1 consistent with Signal 3's own position-relative
# percentile approach, rather than mixing an absolute rule with a relative
# one across the two signals.
final_df['primary_pos'] = final_df['Pos'].str.split(',').str[0]
final_df = final_df[final_df['primary_pos'] != 'GK'].copy()  # excluded per original scope

final_df['value_growth_percentile'] = final_df.groupby('primary_pos')['value_growth_ratio'] \
    .rank(pct=True)
final_df['value_growth_flag'] = final_df['value_growth_percentile'] >= 0.80  # top 20% within position

print(final_df.groupby('primary_pos')['value_growth_ratio'].describe())
print(f"\nFlagged: {final_df['value_growth_flag'].sum()} / {final_df['value_growth_flag'].notna().sum()}")

              count      mean       std  ...  50%       75%     max
primary_pos                              ...                       
DF           1038.0  1.220233  0.986576  ...  1.0  1.466667   9.375
FW            639.0  1.269853  1.212372  ...  1.0  1.500000  13.750
MF            866.0  1.266837  1.264198  ...  1.0  1.500000  25.000

[3 rows x 8 columns]

Flagged: 506 / 2764


## 3. Signal 2 — Club-tier move (data-derived quartiles)

In [5]:
valuations_tm = pd.read_csv(VALUATIONS_FILE)
valuations_tm['date'] = pd.to_datetime(valuations_tm['date'], errors='coerce')
valuations_tm['season_year'] = valuations_tm['date'].dt.year

# IMPORTANT: restrict tiering to clubs actually relevant to this study,
# not the entire global Transfermarkt universe. Quartiling against every
# club worldwide (lower divisions, minor leagues, etc.) puts almost all
# Big-5 clubs at the ceiling tier already at observation, leaving no room
# to detect a genuine within-elite-football rise (e.g. Nantes -> PSG).
relevant_clubs = set(final_df['club_at_observation'].dropna()) | set(final_df['club_2yr_later'].dropna())
print(f"Clubs relevant to this study: {len(relevant_clubs)}")

valuations_relevant = valuations_tm[valuations_tm['current_club_name'].isin(relevant_clubs)]

club_tier_by_season = (
    valuations_relevant.groupby(['current_club_name', 'season_year'])['market_value_in_eur']
    .mean().reset_index().rename(columns={'market_value_in_eur': 'avg_squad_value'})
)

def safe_qcut(x, q=4):
    if x.nunique() < 2:
        return pd.Series([np.nan] * len(x), index=x.index)
    try:
        codes = pd.qcut(x, q, labels=False, duplicates='drop')
        return codes + 1
    except ValueError:
        return pd.Series([np.nan] * len(x), index=x.index)

club_tier_by_season['tier'] = club_tier_by_season.groupby('season_year')['avg_squad_value'] \
    .transform(safe_qcut)

print(f"Rows with a computed tier: {club_tier_by_season['tier'].notna().sum()} / {len(club_tier_by_season)}")
print(club_tier_by_season['tier'].value_counts())

tier_lookup = club_tier_by_season.set_index(['current_club_name', 'season_year'])['tier'].to_dict()

def get_tier(club, year):
    return tier_lookup.get((club, year), None)

final_df['_obs_valuation_year'] = final_df['Season'].apply(lambda s: int(str(s).split('-')[-1]))
final_df['_future_valuation_year'] = final_df['_obs_valuation_year'] + 2

final_df['tier_at_observation'] = final_df.apply(
    lambda r: get_tier(r['club_at_observation'], r['_obs_valuation_year']), axis=1)
final_df['tier_future'] = final_df.apply(
    lambda r: get_tier(r['club_2yr_later'], r['_future_valuation_year']), axis=1)

print("\ntier_at_observation distribution:")
print(final_df['tier_at_observation'].value_counts())
print("\ntier_future distribution:")
print(final_df['tier_future'].value_counts())

final_df['club_tier_flag'] = (
    final_df['tier_future'].astype(float) > final_df['tier_at_observation'].astype(float)
)

print(f"\nFlagged: {final_df['club_tier_flag'].sum()} / {final_df['club_tier_flag'].notna().sum()}")

Clubs relevant to this study: 276
Rows with a computed tier: 5431 / 5432
tier
1.0    1369
4.0    1359
2.0    1355
3.0    1348
Name: count, dtype: int64

tier_at_observation distribution:
tier_at_observation
4.0    1621
3.0     826
2.0     184
1.0      13
Name: count, dtype: int64

tier_future distribution:
tier_future
4.0    1642
3.0     572
2.0     230
1.0     100
Name: count, dtype: int64

Flagged: 327 / 2764


## 4. Signal 3 — Performance breakout (position-specific, top 20th percentile)

**Deliberately uses realized end-product/team-impact stats, not the same expected/process metrics used as model features.** Using `xG90`/`xA90`/`KP_`/`PrgP_` etc. for both the features AND the label would be partially circular — the model could appear predictive just by learning 'high xG now predicts high xG later', which is a fairly trivial pattern, not a genuine test of whether observed process quality predicts realized future output. Realized goals/assists, goal/shot-creating actions, clearances, aerial dominance, and team on/off-pitch impact are a distinct signal family from the underlying-quality features, while still genuinely representing 'breaking out'.

`TklW`, `Int`, `KP_`, `Progression_PrgP/PrgC`, `Clr_`, and `Aerial Duels_Won` are **season totals**, not per-90 rates, in this source — converted explicitly here using `90s`.

**Scope note:** goalkeepers are excluded before constructing the label, consistent with the outfield-player scope of the study. The performance indicators differ by position group, so the composite is not identical across forwards, midfielders and defenders.

In [6]:
final_df['TklW_per90'] = final_df['TklW'] / final_df['90s']
final_df['Int_per90'] = final_df['Int'] / final_df['90s']
final_df['KP_per90'] = final_df['KP_'] / final_df['90s']
final_df['PrgP_per90'] = final_df['Progression_PrgP'] / final_df['90s']
final_df['PrgC_per90'] = final_df['Progression_PrgC'] / final_df['90s']
final_df['AerialWon_per90'] = final_df['Aerial Duels_Won'] / final_df['90s']
final_df['Clr_per90'] = final_df['Clr_'] / final_df['90s']
final_df['GlsAst90'] = final_df['Per 90 Minutes_Gls'] + final_df['Per 90 Minutes_Ast']

# Realized-output composites — deliberately distinct from the feature set
# (xG/xA/progression/tackles/interceptions), which stays in FEATURES for
# modelling in notebook 06 but is NOT reused here for the label.
FW_STATS = ['GlsAst90']
MF_STATS = ['GCA_GCA90', 'SCA_SCA90', 'Team Success_On-Off']
DF_STATS = ['Clr_per90', 'Aerial Duels_Won%', 'Team Success_On-Off']

pos_groups = {'FW': FW_STATS, 'MF': MF_STATS, 'DF': DF_STATS}
final_df_scored = final_df[final_df['primary_pos'].isin(pos_groups.keys())].copy()

def zscore_composite(df, stats, pos_value):
    subset = df[df['primary_pos'] == pos_value].copy()
    missing = [s for s in stats if s not in subset.columns]
    if missing:
        print(f"WARNING for {pos_value}: columns not found, check exact names: {missing}")
    for stat in stats:
        if stat not in subset.columns:
            continue
        mean, std = subset[stat].mean(), subset[stat].std()
        subset[f'_z_{stat}'] = (subset[stat] - mean) / std if std > 0 else 0
    z_cols = [f'_z_{s}' for s in stats if f'_z_{s}' in subset.columns]
    subset['breakout_composite'] = subset[z_cols].sum(axis=1)
    return subset[['Player', 'Season', 'breakout_composite']]

composite_frames = [zscore_composite(final_df_scored, stats, pos) for pos, stats in pos_groups.items()]
composite_df = pd.concat(composite_frames, ignore_index=True)
final_df = final_df.merge(composite_df, on=['Player', 'Season'], how='left')

final_df['breakout_percentile'] = final_df.groupby('primary_pos')['breakout_composite'].rank(pct=True)
final_df['performance_flag'] = final_df['breakout_percentile'] >= 0.80

print(f"Flagged: {final_df['performance_flag'].sum()} / {final_df['performance_flag'].notna().sum()}")

Flagged: 553 / 2764


## 5. Composite label — 2 of 3 signals

In [7]:
final_df['elite_score'] = (
    final_df['value_growth_flag'].fillna(False).astype(int) +
    final_df['club_tier_flag'].fillna(False).astype(int) +
    final_df['performance_flag'].fillna(False).astype(int)
)
final_df['is_elite'] = final_df['elite_score'] >= 2

print("Label counts:")
print(final_df['is_elite'].value_counts())
print(f"\nElite rate: {final_df['is_elite'].mean():.1%}")

print("\nBy position:")
print(final_df.groupby('primary_pos')['is_elite'].mean())

Label counts:
is_elite
False    2515
True      249
Name: count, dtype: int64

Elite rate: 9.0%

By position:
primary_pos
DF    0.095111
FW    0.072569
MF    0.096842
Name: is_elite, dtype: float64


## 6. Sanity check — case-study players

In [8]:
case_study_players = ['Cole Palmer', 'Michael Olise', 'Randal Kolo Muani']
print(final_df[final_df['Player'].isin(case_study_players)][
    ['Player', 'Season', 'value_growth_flag', 'club_tier_flag', 'performance_flag', 'elite_score', 'is_elite']
].to_string(index=False))

           Player    Season  value_growth_flag  club_tier_flag  performance_flag  elite_score  is_elite
Randal Kolo Muani 2020-2021               True            True             False            2      True
Randal Kolo Muani 2021-2022               True            True             False            2      True
    Michael Olise 2022-2023               True           False             False            1     False
Randal Kolo Muani 2022-2023              False           False              True            1     False
      Cole Palmer 2023-2024              False           False              True            1     False
    Michael Olise 2023-2024               True           False              True            2      True
Randal Kolo Muani 2023-2024              False           False              True            1     False
    Michael Olise 2021-2022               True           False             False            1     False


## 7. Save

In [9]:
final_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved labelled dataset: {OUTPUT_FILE}")
print(f"Final dataset shape: {final_df.shape}")

Saved labelled dataset: /mnt/data/football-player-development-prediction-repro/data/processed/final_labeled_dataset.csv
Final dataset shape: (2764, 318)
